In [42]:
import warnings, time, pickle, os
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from mimic import fit_mimic, predict_latent

from __future__ import annotations
from scipy import optimize, stats
from dataclasses import dataclass

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Data exploration

In [30]:
df = pd.read_csv('../data/final_data.csv', index_col=0).set_index('year_quarter')
 
CAUSE_CANDIDATES = {
    'd_tax_burden':                +1,
    'd_direct_tax_share':          +1,
    'd_gov_expenditure_share':     +1,
    'd_fiscal_freedom':            -1,
    'd_rule_of_law':               -1,
    'd_government_effectiveness':  -1,
    'd_corruption_perceptions':    -1,
    'd_business_freedom':          -1,
    'd_economic_freedom':          -1,
    'd_unemployment_rate':         +1,
    'dlog_gdp_per_capita':         -1,
}
INDICATOR_CANDIDATES = {
    'd_cash_demand':            +1,
    'sdiff_deflated_M1':        +1,
    'd_lfp_rate':               -1,
    'd_employment_rate':        -1,
    'dlog_gdp_s21':             -1,
    'sdiff_gdp_2':              -1,
    'dlog_deflated_tax_revenue':-1,
}
all_vars = list(CAUSE_CANDIDATES) + list(INDICATOR_CANDIDATES)
work = df[all_vars].dropna().copy()
print(f"n = {len(work)}", flush=True)
 
 
def safe_one_fit(causes, indicators):
    """One fit, no orientation tries (faster)."""
    try:
        r = fit_mimic(work, causes=list(causes), indicators=list(indicators),
                      standardize=True, n_starts=2, random_state=42)
    except Exception:
        return None
    if not r.converged: return None
    if not np.all(np.isfinite(r.gamma_se)): return None
    if not np.all(np.isfinite(r.lambda_se)): return None
    return r
 
 
def sign_orient(result, ind_signs):
    return -1 if ind_signs[result.indicators[0]] < 0 else +1
 

anchor_indicator_pairs = [
    ('d_cash_demand', 'd_lfp_rate'),
    ('d_cash_demand', 'dlog_gdp_s21'),
    ('sdiff_deflated_M1', 'd_employment_rate'),
]
 
cause_zs = {c: [] for c in CAUSE_CANDIDATES}
cause_signs_correct = {c: 0 for c in CAUSE_CANDIDATES}
cause_attempts = {c: 0 for c in CAUSE_CANDIDATES}
 
t0 = time.time()
for c in CAUSE_CANDIDATES:
    for inds in anchor_indicator_pairs:
        for partner in ['d_unemployment_rate', 'dlog_gdp_per_capita',
                         'd_corruption_perceptions']:
            if partner == c: continue
            r = safe_one_fit((c, partner), inds)
            if r is None: continue
            cause_attempts[c] += 1
            z = abs(r.gamma_z()[0])
            if np.isfinite(z): cause_zs[c].append(z)
            flip = sign_orient(r, INDICATOR_CANDIDATES)
            if np.sign(r.gamma[0] * flip) == np.sign(CAUSE_CANDIDATES[c]):
                cause_signs_correct[c] += 1
    print(f"  c-screen done {c}  ({time.time()-t0:.1f}s)", flush=True)
 
cause_summary = pd.DataFrame({
    'cause': list(CAUSE_CANDIDATES),
    'expected_sign': [CAUSE_CANDIDATES[c] for c in CAUSE_CANDIDATES],
    'n_fits': [cause_attempts[c] for c in CAUSE_CANDIDATES],
    'mean_abs_z': [np.mean(cause_zs[c]) if cause_zs[c] else 0 for c in CAUSE_CANDIDATES],
    'sign_ok_rate': [cause_signs_correct[c] / max(cause_attempts[c],1) for c in CAUSE_CANDIDATES],
}).sort_values('mean_abs_z', ascending=False)
print("\n--- Cause screening ---")
print(cause_summary.to_string(index=False))
cause_summary.to_csv('../outputs/interm_mimic/cause_screening.csv', index=False)
 

print("\nIndicator screening...", flush=True)
ind_zs = {i: [] for i in INDICATOR_CANDIDATES}
ind_attempts = {i: 0 for i in INDICATOR_CANDIDATES}
ind_signs_correct = {i: 0 for i in INDICATOR_CANDIDATES}
anchor_cause_sets = [
    ('d_tax_burden', 'd_rule_of_law', 'd_corruption_perceptions', 'd_unemployment_rate'),
    ('d_gov_expenditure_share', 'd_government_effectiveness', 'd_business_freedom', 'd_unemployment_rate'),
    ('d_tax_burden', 'd_unemployment_rate', 'd_economic_freedom', 'dlog_gdp_per_capita'),
]
t0 = time.time()
for ind in INDICATOR_CANDIDATES:
    for ind2 in INDICATOR_CANDIDATES:
        if ind2 == ind: continue
        for cs in anchor_cause_sets:
            r = safe_one_fit(cs, [ind2, ind])
            if r is None: continue
            ind_attempts[ind] += 1
            z_arr = r.lambda_z()
            z = abs(z_arr[1]) if np.isfinite(z_arr[1]) else 0
            if np.isfinite(z): ind_zs[ind].append(z)
            flip = -1 if INDICATOR_CANDIDATES[ind2] < 0 else +1
            if np.sign(r.lambda_[1] * flip) == np.sign(INDICATOR_CANDIDATES[ind]):
                ind_signs_correct[ind] += 1
    print(f"  i-screen done {ind}  ({time.time()-t0:.1f}s)", flush=True)
 
ind_summary = pd.DataFrame({
    'indicator': list(INDICATOR_CANDIDATES),
    'expected_sign': [INDICATOR_CANDIDATES[i] for i in INDICATOR_CANDIDATES],
    'n_fits': [ind_attempts[i] for i in INDICATOR_CANDIDATES],
    'mean_abs_z': [np.mean(ind_zs[i]) if ind_zs[i] else 0 for i in INDICATOR_CANDIDATES],
    'sign_ok_rate': [ind_signs_correct[i] / max(ind_attempts[i],1) for i in INDICATOR_CANDIDATES],
}).sort_values('mean_abs_z', ascending=False)
print("\n--- Indicator screening ---")
print(ind_summary.to_string(index=False))
ind_summary.to_csv('../outputs/interm_mimic/indicator_screening.csv', index=False)
 
 
short_causes = cause_summary[
    (cause_summary['mean_abs_z'] >= cause_summary['mean_abs_z'].median())
    | (cause_summary['sign_ok_rate'] >= 0.6)
]['cause'].tolist()
if len(short_causes) < 7: short_causes = cause_summary['cause'].head(7).tolist()
short_inds = ind_summary[
    (ind_summary['mean_abs_z'] >= ind_summary['mean_abs_z'].median())
    | (ind_summary['sign_ok_rate'] >= 0.6)
]['indicator'].tolist()
if len(short_inds) < 5: short_inds = ind_summary['indicator'].head(5).tolist()
 
print(f"\nShortlist causes ({len(short_causes)}): {short_causes}")
print(f"Shortlist indicators ({len(short_inds)}): {short_inds}")
 
with open('../outputs/interm_mimic/shortlists.pkl', 'wb') as f:
    pickle.dump({'causes': short_causes, 'indicators': short_inds,
                 'CAUSE_CANDIDATES': CAUSE_CANDIDATES,
                 'INDICATOR_CANDIDATES': INDICATOR_CANDIDATES}, f)
print("Saved shortlists.pkl")

n = 44
  c-screen done d_tax_burden  (0.4s)
  c-screen done d_direct_tax_share  (0.8s)
  c-screen done d_gov_expenditure_share  (1.2s)
  c-screen done d_fiscal_freedom  (1.6s)
  c-screen done d_rule_of_law  (2.0s)
  c-screen done d_government_effectiveness  (2.4s)
  c-screen done d_corruption_perceptions  (2.8s)
  c-screen done d_business_freedom  (3.3s)
  c-screen done d_economic_freedom  (3.8s)
  c-screen done d_unemployment_rate  (4.3s)
  c-screen done dlog_gdp_per_capita  (4.7s)

--- Cause screening ---
                     cause  expected_sign  n_fits  mean_abs_z  sign_ok_rate
       dlog_gdp_per_capita             -1       3    4.093714      1.000000
              d_tax_burden              1       4    2.298807      0.250000
        d_direct_tax_share              1       4    1.449636      0.250000
       d_unemployment_rate              1       1    1.405117      1.000000
  d_corruption_perceptions             -1       3    1.358493      0.666667
d_government_effectiveness     

# Specification search

In [31]:
NC_RANGE = [3, 4, 5]
NI_RANGE = [2, 3]
 
with open('../outputs/interm_mimic/shortlists.pkl', 'rb') as f:
    sl = pickle.load(f)
 
CAUSE_CANDIDATES = sl['CAUSE_CANDIDATES']
INDICATOR_CANDIDATES = sl['INDICATOR_CANDIDATES']
short_causes = sl['causes'][:6]
short_inds = sl['indicators'][:5]
 
df = pd.read_csv('../data/final_data.csv', index_col=0).set_index('year_quarter')
all_vars = list(set(short_causes + short_inds))
work = df[all_vars].dropna().copy()
 
 
def sign_orient(result, ind_signs):
    return -1 if ind_signs[result.indicators[0]] < 0 else +1
 
 
def n_correct_signs(result, cause_signs, ind_signs):
    flip = sign_orient(result, ind_signs)
    g = result.gamma * flip
    lam = result.lambda_ * flip
    nc = sum(np.sign(g[i]) == np.sign(cause_signs[c])
             for i, c in enumerate(result.causes))
    nl = sum(np.sign(lam[i]) == np.sign(ind_signs[result.indicators[i]])
             for i in range(1, len(result.indicators)))
    return nc, nl
 
 
def quick_score(result, cause_signs, ind_signs, alpha=0.10):
    nq = len(result.causes); npi = len(result.indicators)
    pg = result.gamma_p()
    pl = result.lambda_p()
    n_sig_g = int(np.sum(pg < alpha))
    n_sig_l = int(np.sum(pl[1:] < alpha)) if npi > 1 else 0
    nc, nl = n_correct_signs(result, cause_signs, ind_signs)
    mean_abs_z = float(np.mean(np.abs(result.gamma_z())))
    score = (
        2.0 * (n_sig_g / nq)
      + 1.5 * (n_sig_l / max(npi - 1, 1))
      + 1.5 * (nc / nq)
      + 1.0 * (nl / max(npi - 1, 1))
      + 0.5 * (1.0 if (not np.isnan(result.cfi) and result.cfi > 0.90) else 0.0)
      + 0.3 * (1.0 if (not np.isnan(result.rmsea) and result.rmsea < 0.10) else 0.0)
      + 0.2 * (1.0 if result.pvalue > 0.05 else 0.0)
      + 0.1 * mean_abs_z
    )
    return dict(n_sig_g=n_sig_g, n_sig_l=n_sig_l, nc=nc, nl=nl,
                mean_abs_z=mean_abs_z, score=score)
 
 
def safe_fit(causes, indicators):
    best = None
    for ref_idx in range(len(indicators)):
        inds = [indicators[ref_idx]] + [x for j,x in enumerate(indicators) if j != ref_idx]
        try:
            r = fit_mimic(work, causes=list(causes), indicators=inds,
                          standardize=True, n_starts=1, random_state=42)
        except Exception:
            continue
        if not r.converged: continue
        if not np.all(np.isfinite(r.gamma_se)): continue
        if not np.all(np.isfinite(r.lambda_se)): continue
        sc = quick_score(r, CAUSE_CANDIDATES, INDICATOR_CANDIDATES)
        if best is None or sc['score'] > best[1]['score']:
            best = (r, sc, inds)
    return best
 
 
for NC in NC_RANGE:
    for NI in NI_RANGE:
        total = (sum(1 for _ in itertools.combinations(short_causes, NC))
               * sum(1 for _ in itertools.combinations(short_inds, NI)))
        print(f"\n{'='*60}", flush=True)
        print(f"Chunk NC={NC} NI={NI}: {total} specs", flush=True)
        print(f"{'='*60}", flush=True)
 
        results = []
        count = 0
        t0 = time.time()
        for causes in itertools.combinations(short_causes, NC):
            for inds in itertools.combinations(short_inds, NI):
                count += 1
                best = safe_fit(causes, list(inds))
                if best is None: continue
                r, sc, inds_ordered = best
                results.append({
                    'causes': list(causes),
                    'indicators': list(inds_ordered),
                    'n_causes': NC, 'n_indicators': NI,
                    'n_sig_causes': sc['n_sig_g'],
                    'n_sig_inds': sc['n_sig_l'],
                    'correct_g': sc['nc'], 'correct_l': sc['nl'],
                    'mean_abs_z': sc['mean_abs_z'],
                    'chi2': r.chi2, 'df': r.df, 'pvalue': r.pvalue,
                    'cfi': r.cfi, 'tli': r.tli, 'rmsea': r.rmsea,
                    'aic': r.aic, 'bic': r.bic,
                    'score': sc['score'],
                    'result_obj': r,
                })
                if count % 50 == 0:
                    print(f"  {count}/{total}  ({time.time()-t0:.0f}s, {len(results)} ok)", flush=True)
 
        print(f"Chunk NC={NC} NI={NI} done: {len(results)}/{total} in {time.time()-t0:.0f}s", flush=True)
 
        out_path = f'../outputs/interm_mimic/stage_b_{NC}_{NI}.pkl'
        with open(out_path, 'wb') as f:
            pickle.dump(results, f)
        print(f"Saved {out_path}", flush=True)
 
print("\nAll chunks complete.")


Chunk NC=3 NI=2: 200 specs
  50/200  (4s, 45 ok)
  100/200  (5s, 83 ok)
  150/200  (7s, 122 ok)
  200/200  (9s, 165 ok)
Chunk NC=3 NI=2 done: 165/200 in 9s
Saved ../outputs/interm_mimic/stage_b_3_2.pkl

Chunk NC=3 NI=3: 200 specs
  50/200  (3s, 46 ok)
  100/200  (6s, 95 ok)
  150/200  (9s, 145 ok)
  200/200  (13s, 191 ok)
Chunk NC=3 NI=3 done: 191/200 in 13s
Saved ../outputs/interm_mimic/stage_b_3_3.pkl

Chunk NC=4 NI=2: 150 specs
  50/150  (2s, 38 ok)
  100/150  (4s, 82 ok)
  150/150  (6s, 126 ok)
Chunk NC=4 NI=2 done: 126/150 in 6s
Saved ../outputs/interm_mimic/stage_b_4_2.pkl

Chunk NC=4 NI=3: 150 specs
  50/150  (3s, 47 ok)
  100/150  (7s, 96 ok)
  150/150  (10s, 141 ok)
Chunk NC=4 NI=3 done: 141/150 in 10s
Saved ../outputs/interm_mimic/stage_b_4_3.pkl

Chunk NC=5 NI=2: 60 specs
  50/60  (3s, 36 ok)
Chunk NC=5 NI=2 done: 44/60 in 3s
Saved ../outputs/interm_mimic/stage_b_5_2.pkl

Chunk NC=5 NI=3: 60 specs
  50/60  (4s, 45 ok)
Chunk NC=5 NI=3 done: 55/60 in 5s
Saved ../outputs/inter

# Consolidate & rank

In [34]:
ROOT = '../outputs/interm_mimic'
 
all_results = []
for fn in sorted(os.listdir(ROOT)):
    if fn.startswith('stage_b_') and fn.endswith('.pkl') and 'nc' not in fn:
        with open(os.path.join(ROOT, fn), 'rb') as f:
            chunk = pickle.load(f)
        all_results.extend(chunk)
        print(f"  {fn}: +{len(chunk)} -> total {len(all_results)}")
 
print(f"\nTotal converged specifications: {len(all_results)}")
 
results_sorted = sorted(all_results, key=lambda r: -r['score'])
 
out = pd.DataFrame([{k: v for k, v in r.items() if k != 'result_obj'}
                    for r in results_sorted])
out['causes_str'] = out['causes'].apply(lambda L: '|'.join(L))
out['indicators_str'] = out['indicators'].apply(lambda L: '|'.join(L))
csv_cols = ['n_causes','n_indicators','n_sig_causes','n_sig_inds',
            'correct_g','correct_l','mean_abs_z',
            'chi2','df','pvalue','cfi','tli','rmsea','aic','bic',
            'score','causes_str','indicators_str']
out[csv_cols].to_csv(os.path.join(ROOT, 'all_specifications_ranked.csv'), index=False)
print(f"Saved all_specifications_ranked.csv ({len(out)} rows)")
 
with open(os.path.join(ROOT, 'top_results.pkl'), 'wb') as f:
    pickle.dump(results_sorted[:20], f)
 
print("\n" + "=" * 78)
print("TOP 15 SPECIFICATIONS BY COMPOSITE SCORE")
print("=" * 78)
hdr = f"{'#':>3} {'score':>5} {'sigC':>6} {'sigI':>6} {'okG':>6} {'okL':>6} " \
      f"{'CFI':>5} {'RMSEA':>5} {'p':>5} {'AIC':>7}"
print(hdr)
for i, r in enumerate(results_sorted[:15]):
    print(f"{i+1:>3} {r['score']:>5.2f} "
          f"{r['n_sig_causes']}/{r['n_causes']:<4} "
          f"{r['n_sig_inds']}/{r['n_indicators']-1:<4} "
          f"{r['correct_g']}/{r['n_causes']:<4} "
          f"{r['correct_l']}/{r['n_indicators']-1:<4} "
          f"{r['cfi']:>5.2f} {r['rmsea']:>5.2f} {r['pvalue']:>5.2f} "
          f"{r['aic']:>7.1f}")
    print(f"      C: {r['causes']}")
    print(f"      I: {r['indicators']}")
 

print("\n" + "=" * 78)
print("TOP 5 BY BIC (most parsimonious model with good fit)")
print("=" * 78)
bic_sorted = sorted([r for r in all_results
                     if r['cfi'] > 0.85 and r['n_sig_causes'] >= 2
                     and r['correct_g'] >= r['n_causes'] - 1],
                    key=lambda r: r['bic'])
for i, r in enumerate(bic_sorted[:5]):
    print(f"{i+1}. BIC={r['bic']:.2f}  AIC={r['aic']:.2f}  CFI={r['cfi']:.3f}  RMSEA={r['rmsea']:.3f}")
    print(f"   sigC={r['n_sig_causes']}/{r['n_causes']}  signs ok={r['correct_g']}/{r['n_causes']}")
    print(f"   C: {r['causes']}")
    print(f"   I: {r['indicators']}")

  stage_b_3_2.pkl: +165 -> total 165
  stage_b_3_3.pkl: +191 -> total 356
  stage_b_4_2.pkl: +126 -> total 482
  stage_b_4_3.pkl: +141 -> total 623
  stage_b_5_2.pkl: +44 -> total 667
  stage_b_5_3.pkl: +55 -> total 722

Total converged specifications: 722
Saved all_specifications_ranked.csv (722 rows)

TOP 15 SPECIFICATIONS BY COMPOSITE SCORE
  # score   sigC   sigI    okG    okL   CFI RMSEA     p     AIC
  1  6.48 2/3    1/1    3/3    1/1     1.00  0.00  0.40   612.8
      C: ['dlog_gdp_per_capita', 'd_tax_burden', 'd_corruption_perceptions']
      I: ['sdiff_gdp_2', 'd_employment_rate']
  2  6.42 3/3    1/1    2/3    1/1     0.91  0.18  0.09   598.4
      C: ['d_direct_tax_share', 'd_corruption_perceptions', 'd_government_effectiveness']
      I: ['d_employment_rate', 'd_lfp_rate']
  3  6.16 2/3    1/1    2/3    1/1     1.00  0.03  0.35   589.5
      C: ['d_tax_burden', 'd_unemployment_rate', 'd_corruption_perceptions']
      I: ['dlog_deflated_tax_revenue', 'sdiff_gdp_2']
  4  6.10

# Final selection

In [35]:
ROOT = '../outputs/interm_mimic'
 
CAUSE_CANDIDATES = {
    'd_tax_burden': +1, 'd_direct_tax_share': +1, 'd_gov_expenditure_share': +1,
    'd_fiscal_freedom': -1, 'd_rule_of_law': -1, 'd_government_effectiveness': -1,
    'd_corruption_perceptions': -1, 'd_business_freedom': -1, 'd_economic_freedom': -1,
    'd_unemployment_rate': +1, 'dlog_gdp_per_capita': -1,
}
INDICATOR_CANDIDATES = {
    'd_cash_demand': +1, 'sdiff_deflated_M1': +1, 'd_lfp_rate': -1,
    'd_employment_rate': -1, 'dlog_gdp_s21': -1, 'sdiff_gdp_2': -1,
    'dlog_deflated_tax_revenue': -1,
}
GDP_INDICATORS = {'dlog_gdp_s21', 'sdiff_gdp_2'}
GDP_CAUSES = {'dlog_gdp_per_capita'}

all_results = []
for fn in sorted(os.listdir(ROOT)):
    if fn.startswith('stage_b_') and fn.endswith('.pkl') and 'nc' not in fn:
        with open(os.path.join(ROOT, fn), 'rb') as f:
            all_results.extend(pickle.load(f))
print(f"Total fitted specs: {len(all_results)}")

def is_clean(r):
    has_gdp_cause = any(c in GDP_CAUSES for c in r['causes'])
    has_gdp_ind   = any(i in GDP_INDICATORS for i in r['indicators'])
    return not (has_gdp_cause and has_gdp_ind)
 
clean = [r for r in all_results if is_clean(r)]
print(f"After dropping GDP double-counting specs: {len(clean)}")

def reorient_score(rec, alpha=0.10):
    res = rec['result_obj']
    nq = len(res.causes); npi = len(res.indicators)
    pg = res.gamma_p()
    pl = res.lambda_p()
    n_sig_g = int(np.sum(pg < alpha))
    n_sig_l = int(np.sum(pl[1:] < alpha)) if npi > 1 else 0
 
    best_flip, best_nl = None, -1
    for flip in [+1, -1]:
        lam = res.lambda_ * flip
        nl = sum(np.sign(lam[i]) == np.sign(INDICATOR_CANDIDATES[res.indicators[i]])
                 for i in range(npi))
        if nl > best_nl:
            best_nl = nl; best_flip = flip
    flip = best_flip
    g_or = res.gamma * flip
    lam_or = res.lambda_ * flip
    nc = sum(np.sign(g_or[i]) == np.sign(CAUSE_CANDIDATES[c])
             for i, c in enumerate(res.causes))
    nl = best_nl
    mean_abs_z = float(np.mean(np.abs(res.gamma_z())))
    score = (
        2.0 * (n_sig_g / nq)
      + 1.5 * (n_sig_l / max(npi - 1, 1))
      + 1.5 * (nc / nq)
      + 1.0 * (nl / npi)
      + 0.5 * (1.0 if (not np.isnan(res.cfi) and res.cfi > 0.90) else 0.0)
      + 0.3 * (1.0 if (not np.isnan(res.rmsea) and res.rmsea < 0.10) else 0.0)
      + 0.2 * (1.0 if res.pvalue > 0.05 else 0.0)
      + 0.1 * mean_abs_z
    )
    return {'score': score, 'n_sig_g': n_sig_g, 'n_sig_l': n_sig_l,
            'nc': nc, 'nl': nl, 'flip': flip, 'mean_abs_z': mean_abs_z}
 
for r in clean:
    s = reorient_score(r)
    r['score'] = s['score']
    r['n_sig_causes'] = s['n_sig_g']
    r['n_sig_inds'] = s['n_sig_l']
    r['correct_g'] = s['nc']
    r['correct_l'] = s['nl']
    r['flip'] = s['flip']
    r['mean_abs_z'] = s['mean_abs_z']
 
clean.sort(key=lambda r: -r['score'])
 
print("\n=== TOP 10 CLEAN SPECIFICATIONS ===")
hdr = f"{'#':>3} {'score':>5} {'sigC':>6} {'sigI':>6} {'okG':>6} {'okL':>6} " \
      f"{'CFI':>5} {'RMSEA':>5} {'p':>5} {'BIC':>7}"
print(hdr)
for i, r in enumerate(clean[:10]):
    print(f"{i+1:>3} {r['score']:>5.2f} "
          f"{r['n_sig_causes']}/{r['n_causes']:<4} "
          f"{r['n_sig_inds']}/{r['n_indicators']-1:<4} "
          f"{r['correct_g']}/{r['n_causes']:<4} "
          f"{r['correct_l']}/{r['n_indicators']:<4} "
          f"{r['cfi']:>5.2f} {r['rmsea']:>5.2f} {r['pvalue']:>5.2f} "
          f"{r['bic']:>7.1f}")
    print(f"      C: {r['causes']}")
    print(f"      I: {r['indicators']}")

out = pd.DataFrame([{
    'n_causes': r['n_causes'], 'n_indicators': r['n_indicators'],
    'n_sig_causes': r['n_sig_causes'], 'n_sig_inds': r['n_sig_inds'],
    'correct_g': r['correct_g'], 'correct_l': r['correct_l'],
    'mean_abs_z': r['mean_abs_z'],
    'chi2': r['chi2'], 'df': r['df'], 'pvalue': r['pvalue'],
    'cfi': r['cfi'], 'tli': r['tli'], 'rmsea': r['rmsea'],
    'aic': r['aic'], 'bic': r['bic'], 'score': r['score'],
    'causes': '|'.join(r['causes']),
    'indicators': '|'.join(r['indicators']),
} for r in clean])
out.to_csv(os.path.join(ROOT, 'final_specifications_ranked.csv'), index=False)
print(f"\nSaved final_specifications_ranked.csv ({len(out)} rows)")

WINNER = clean[0]
print(f"\n=== WINNING SPECIFICATION ===")
print(f"Causes    : {WINNER['causes']}")
print(f"Indicators: {WINNER['indicators']}")

df = pd.read_csv('../data/final_data.csv', index_col=0)
df = df.set_index('year_quarter')
work = df[WINNER['causes'] + WINNER['indicators']].dropna().copy()
print(f"\nObservations used: {len(work)}")
 
final = fit_mimic(work, causes=WINNER['causes'], indicators=WINNER['indicators'],
                  standardize=True, n_starts=10, random_state=0)
print(f"Converged: {final.converged}")

flip = WINNER['flip']
gamma_o = final.gamma * flip
lambda_o = final.lambda_ * flip

print("\n" + "=" * 78)
print("FINAL MIMIC ESTIMATES (oriented so that 'eta up = more shadow economy')")
print("=" * 78)
print("\nStructural equation (causes -> latent shadow economy)")
print(f"{'variable':<32}{'estimate':>10}{'std.err':>10}{'z':>8}{'p-value':>10}{'expect':>8}{'sign':>6}")
for i, c in enumerate(final.causes):
    z = gamma_o[i] / final.gamma_se[i]
    from scipy.stats import norm
    p = 2 * (1 - norm.cdf(abs(z)))
    sign_ok = '+' if np.sign(gamma_o[i]) == np.sign(CAUSE_CANDIDATES[c]) else '-'
    print(f"{c:<32}{gamma_o[i]:>+10.4f}{final.gamma_se[i]:>10.4f}{z:>+8.2f}{p:>10.4f}"
          f"{('+' if CAUSE_CANDIDATES[c] > 0 else '-'):>8}{sign_ok:>6}")
 
print("\nMeasurement equation (latent -> indicators)")
print(f"{'variable':<32}{'estimate':>10}{'std.err':>10}{'z':>8}{'p-value':>10}{'expect':>8}{'sign':>6}")
for i, ind in enumerate(final.indicators):
    if i == 0:
        sign_ok = '+' if np.sign(lambda_o[i]) == np.sign(INDICATOR_CANDIDATES[ind]) else '-'
        print(f"{ind:<32}{lambda_o[i]:>+10.4f}{'(fixed)':>10}{'-':>8}{'-':>10}"
              f"{('+' if INDICATOR_CANDIDATES[ind] > 0 else '-'):>8}{sign_ok:>6}")
    else:
        z = lambda_o[i] / final.lambda_se[i]
        from scipy.stats import norm
        p = 2 * (1 - norm.cdf(abs(z)))
        sign_ok = '+' if np.sign(lambda_o[i]) == np.sign(INDICATOR_CANDIDATES[ind]) else '-'
        print(f"{ind:<32}{lambda_o[i]:>+10.4f}{final.lambda_se[i]:>10.4f}{z:>+8.2f}{p:>10.4f}"
              f"{('+' if INDICATOR_CANDIDATES[ind] > 0 else '-'):>8}{sign_ok:>6}")
 
print("\nVariance components:")
print(f"  psi (struct. disturbance): {final.psi:.4f}")
print(f"  theta (meas. errors):      {dict(zip(final.indicators, np.round(final.theta, 4)))}")
 
print("\nFit indices:")
print(f"  Chi-squared : {final.chi2:.3f} (df={final.df})  p-value: {final.pvalue:.4f}")
print(f"  CFI         : {final.cfi:.4f}")
print(f"  TLI         : {final.tli:.4f}")
print(f"  RMSEA       : {final.rmsea:.4f}")
print(f"  Log-likelihood: {final.loglik:.2f}")
print(f"  AIC         : {final.aic:.2f}")
print(f"  BIC         : {final.bic:.2f}")
print(f"  N           : {final.n_obs}")

work_for_eta = df[final.causes].copy().dropna()
X = work_for_eta.values
X = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
eta_z = (X @ (final.gamma * flip)) 
eta_series = pd.Series(eta_z, index=work_for_eta.index, name='eta_z')
 
BENCH_QUARTER = '2015Q4'
BENCH_VALUE   = 44.8
if BENCH_QUARTER in eta_series.index:
    e0 = eta_series[BENCH_QUARTER]
else:
    e0 = eta_series.iloc[-1]

eta_index = (eta_series - eta_series.min()) / (eta_series[BENCH_QUARTER] - eta_series.min()) * BENCH_VALUE
 
ts = pd.DataFrame({
    'eta_z':                eta_series,
    'shadow_pct_gdp_estim': eta_index,
})
ts.to_csv(os.path.join(ROOT, 'shadow_economy_estimates.csv'))
print(f"\nLatent index saved to shadow_economy_estimates.csv")
print(f"\n  Mean estimate over 2010-2021: {eta_index.mean():.1f}% of GDP")
print(f"  Min:   {eta_index.min():.1f}% ({eta_index.idxmin()})")
print(f"  Max:   {eta_index.max():.1f}% ({eta_index.idxmax()})")
print(f"  2015 (benchmark): {eta_index['2015Q4']:.1f}%")
 

with open(os.path.join(ROOT, 'final_fit.pkl'), 'wb') as f:
    pickle.dump({'result': final, 'flip': flip,
                 'gamma_oriented': gamma_o, 'lambda_oriented': lambda_o,
                 'eta_z': eta_series, 'eta_pct_gdp': eta_index,
                 'winner_record': WINNER}, f)
print("Saved final_fit.pkl")


Total fitted specs: 722
After dropping GDP double-counting specs: 492

=== TOP 10 CLEAN SPECIFICATIONS ===
  # score   sigC   sigI    okG    okL   CFI RMSEA     p     BIC
  1  6.42 3/3    1/1    2/3    2/2     0.91  0.18  0.09   610.9
      C: ['d_direct_tax_share', 'd_corruption_perceptions', 'd_government_effectiveness']
      I: ['d_employment_rate', 'd_lfp_rate']
  2  6.16 2/3    1/1    2/3    2/2     1.00  0.03  0.35   602.0
      C: ['d_tax_burden', 'd_unemployment_rate', 'd_corruption_perceptions']
      I: ['dlog_deflated_tax_revenue', 'sdiff_gdp_2']
  3  6.10 2/3    1/1    2/3    2/2     1.00  0.00  0.43   603.3
      C: ['d_tax_burden', 'd_unemployment_rate', 'd_government_effectiveness']
      I: ['dlog_deflated_tax_revenue', 'sdiff_gdp_2']
  4  6.03 3/4    1/1    3/4    2/2     0.92  0.13  0.15   733.4
      C: ['dlog_gdp_per_capita', 'd_direct_tax_share', 'd_corruption_perceptions', 'd_government_effectiveness']
      I: ['d_employment_rate', 'd_lfp_rate']
  5  6.01 3/4   

# Calibration

In [36]:
with open('../outputs/interm_mimic/final_fit.pkl', 'rb') as f:
    fit = pickle.load(f)
 
eta_z = fit['eta_z']
print(f"eta_z (changes) over {eta_z.index.min()} - {eta_z.index.max()}")
 
eta_level = eta_z.cumsum()

ANCHOR_PERIOD = ('2014Q1', '2016Q4')
ANCHOR_VALUE  = 44.8
 
mask = (eta_level.index >= ANCHOR_PERIOD[0]) & (eta_level.index <= ANCHOR_PERIOD[1])
anchor_mean_z = eta_level[mask].mean()

target_std_pp = 5.0
sigma_z = eta_level.std()
if sigma_z > 0:

    scaled = (eta_level - anchor_mean_z) / sigma_z * target_std_pp + ANCHOR_VALUE
else:
    scaled = eta_level * 0 + ANCHOR_VALUE
 
shadow_pct_gdp = scaled.copy()
shadow_pct_gdp.name = 'shadow_pct_gdp'
 
shadow_smoothed = shadow_pct_gdp.rolling(4, min_periods=1).mean()
shadow_smoothed.name = 'shadow_pct_gdp_4q_avg'
 
ts = pd.concat([eta_z.rename('eta_z_change'),
                eta_level.rename('eta_level'),
                shadow_pct_gdp,
                shadow_smoothed], axis=1)
ts.index.name = 'year_quarter'
 
print("\n=== Shadow economy time series (% of GDP) ===")
print(f"Mean  : {shadow_pct_gdp.mean():.1f}%")
print(f"Median: {shadow_pct_gdp.median():.1f}%")
print(f"Stdev : {shadow_pct_gdp.std():.1f}pp")
print(f"Range : [{shadow_pct_gdp.min():.1f}%, {shadow_pct_gdp.max():.1f}%]")
print(f"Anchor mean over {ANCHOR_PERIOD}: {shadow_pct_gdp[mask].mean():.2f}% (target: {ANCHOR_VALUE}%)")
 
print("\n--- 4-quarter rolling average (trend) ---")
print(shadow_smoothed.round(1).to_string())
 
ts.to_csv('../outputs/interm_mimic/shadow_economy_estimates.csv')
print(f"\nSaved ../outputs/shadow_economy_estimates.csv ({len(ts)} rows)")
 

eta_z (changes) over 2010Q2 - 2021Q4

=== Shadow economy time series (% of GDP) ===
Mean  : 41.2%
Median: 39.9%
Stdev : 5.0pp
Range : [34.6%, 53.2%]
Anchor mean over ('2014Q1', '2016Q4'): 44.80% (target: 44.8%)

--- 4-quarter rolling average (trend) ---
year_quarter
2010Q2    41.6
2010Q3    41.5
2010Q4    41.7
2011Q1    40.4
2011Q2    39.4
2011Q3    38.1
2011Q4    37.1
2012Q1    36.8
2012Q2    36.1
2012Q3    35.7
2012Q4    35.1
2013Q1    36.0
2013Q2    36.8
2013Q3    37.7
2013Q4    38.4
2014Q1    41.9
2014Q2    44.7
2014Q3    47.8
2014Q4    50.5
2015Q1    48.3
2015Q2    47.2
2015Q3    45.7
2015Q4    45.1
2016Q1    43.8
2016Q2    41.8
2016Q3    40.5
2016Q4    38.7
2017Q1    38.4
2017Q2    38.8
2017Q3    38.7
2017Q4    38.2
2018Q1    39.0
2018Q2    39.4
2018Q3    39.7
2018Q4    40.3
2019Q1    42.8
2019Q2    45.4
2019Q3    47.8
2019Q4    50.4
2020Q1    47.4
2020Q2    44.1
2020Q3    40.6
2020Q4    37.5
2021Q1    38.0
2021Q2    39.0
2021Q3    40.1
2021Q4    41.0

Saved ../outputs/shadow_eco

# Plotting

In [38]:
ts = pd.read_csv('../outputs/interm_mimic/shadow_economy_estimates.csv', index_col=0)
 
def quarter_to_date(q):
    y, qn = q.split('Q')
    month = {'1': 2, '2': 5, '3': 8, '4': 11}[qn]
    return pd.Timestamp(year=int(y), month=month, day=15)
 
ts.index = [quarter_to_date(q) for q in ts.index]
 
fig, ax = plt.subplots(figsize=(11, 6))
 
ax.plot(ts.index, ts['shadow_pct_gdp'], color='#9aaab8', lw=1.0,
        label='Quarterly estimate', alpha=0.8, marker='o', markersize=3)
ax.plot(ts.index, ts['shadow_pct_gdp_4q_avg'], color='#1f4e79', lw=2.4,
        label='4-quarter moving average')

ax.axvspan(pd.Timestamp(2014, 2, 1), pd.Timestamp(2015, 12, 1),
           alpha=0.10, color='red', zorder=0)
ax.text(pd.Timestamp(2014, 9, 1), 52, '2014–15\nUkraine crisis',
        ha='center', fontsize=9, color='darkred')
 
ax.axvspan(pd.Timestamp(2020, 2, 1), pd.Timestamp(2020, 9, 1),
           alpha=0.10, color='orange', zorder=0)
ax.text(pd.Timestamp(2020, 5, 1), 52, 'COVID-19',
        ha='center', fontsize=9, color='darkorange')
 
ax.axhline(44.8, ls='--', lw=0.8, color='#444')
ax.text(pd.Timestamp(2010, 5, 1), 45.4,
        'Anchor: 44.8% (Medina-Schneider 2018, mean 2014–16)',
        fontsize=8, color='#333')
 
ax.set_ylabel('Shadow economy (% of GDP)', fontsize=11)
ax.set_xlabel('')
ax.set_title('Estimated Shadow Economy of Ukraine, 2010Q1–2021Q4 (MIMIC model)',
             fontsize=12, pad=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_ylim(28, 55)
ax.grid(alpha=0.3, axis='y')
ax.legend(loc='lower right', fontsize=10, frameon=True)
 
plt.tight_layout()
plt.savefig('../outputs/interm_mimic/shadow_economy_chart.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved shadow_economy_chart.png')

Saved shadow_economy_chart.png


# Testing various anchors

In [41]:
with open('../outputs/interm_mimic/final_fit.pkl', 'rb') as f:
    fit = pickle.load(f)

eta_z = fit['eta_z']
eta_level = eta_z.cumsum()
sigma_z = eta_level.std()
TARGET_STD_PP = 5.0

anchors = [
    ('KIIS_2017_18',  'KIIS 2017–2018 (38.4%)',        '2017Q1', '2018Q4', 38.4),
    ('NBU_2018',      'NBU 2018 (23.8%)',              '2018Q1', '2018Q4', 23.8),
    ('HS_2010',       'Hassan & Schneider 2010 (59.49%)', '2010Q2', '2010Q4', 59.49),
    ('HS_2011',       'Hassan & Schneider 2011 (53.60%)', '2011Q1', '2011Q4', 53.60),
    ('HS_2012',       'Hassan & Schneider 2012 (54.24%)', '2012Q1', '2012Q4', 54.24),
    ('HS_2013',       'Hassan & Schneider 2013 (53.53%)', '2013Q1', '2013Q4', 53.53),
]


def rescale(eta_level, period_start, period_end, target_value,
            target_std=TARGET_STD_PP):
    mask = (eta_level.index >= period_start) & (eta_level.index <= period_end)
    if not mask.any():
        raise ValueError(f"Anchor period {period_start}-{period_end} not in series.")
    anchor_mean = eta_level[mask].mean()
    return (eta_level - anchor_mean) / sigma_z * target_std + target_value


out = pd.DataFrame(index=eta_level.index)
out['eta_z_change'] = eta_z
out['eta_level'] = eta_level

for short, label, p_start, p_end, target in anchors:
    series = rescale(eta_level, p_start, p_end, target)
    out[f'{short}_q'] = series
    out[f'{short}_4qma'] = series.rolling(4, min_periods=1).mean()

out.index.name = 'year_quarter'
out.to_csv('../outputs/interm_mimic/shadow_economy_anchors.csv')
print(f"Saved shadow_economy_anchors.csv  ({len(out)} rows, {len(out.columns)} cols)")

# Per-anchor summary
print("\n=== Summary by anchor (quarterly series) ===")
for short, label, p_start, p_end, target in anchors:
    s = out[f'{short}_q']
    mask = (s.index >= p_start) & (s.index <= p_end)
    achieved = s[mask].mean()
    print(f"  {label}")
    print(f"    period {p_start}-{p_end}: target {target:.2f}%, achieved {achieved:.2f}%")
    print(f"    full-period mean  : {s.mean():.1f}%")
    print(f"    full-period range : [{s.min():.1f}%, {s.max():.1f}%]")

def quarter_to_date(q):
    y, qn = q.split('Q')
    month = {'1': 2, '2': 5, '3': 8, '4': 11}[qn]
    return pd.Timestamp(year=int(y), month=month, day=15)

dates = pd.DatetimeIndex([quarter_to_date(q) for q in out.index])

fig, axes = plt.subplots(2, 1, figsize=(12, 10),
                         gridspec_kw={'height_ratios': [2, 1]})

ax = axes[0]
colors = ['#1f4e79', '#c0504d', '#4f81bd', '#9bbb59',
          '#8064a2', '#f79646', '#2c4d7c']
for i, (short, label, p_start, p_end, target) in enumerate(anchors):
    series = out[f'{short}_4qma'].values
    ax.plot(dates, series, color=colors[i % len(colors)], lw=2.0,
            label=f'{label}', alpha=0.9)

    s_mask = (out.index >= p_start) & (out.index <= p_end)
    if s_mask.any():
        anchor_dates = dates[s_mask]
        ax.plot(anchor_dates, [target] * len(anchor_dates),
                color=colors[i % len(colors)], lw=4, alpha=0.45,
                solid_capstyle='butt')

ax.set_ylabel('Shadow economy (% of GDP)\n4-quarter moving average', fontsize=10)
ax.set_title('Ukraine shadow economy estimates under different anchor benchmarks',
             fontsize=12, pad=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.grid(alpha=0.3, axis='y')
ax.legend(loc='upper right', fontsize=8, frameon=True, ncol=1)
ax.set_xlim(dates.min(), dates.max())
ax.axvspan(pd.Timestamp(2014, 2, 1), pd.Timestamp(2015, 12, 1),
           alpha=0.07, color='red', zorder=0)
ax.axvspan(pd.Timestamp(2020, 2, 1), pd.Timestamp(2020, 9, 1),
           alpha=0.07, color='orange', zorder=0)

ax2 = axes[1]
positions = np.arange(len(anchors))
mins, maxs, means, targets = [], [], [], []
short_labels = []
for short, label, p_start, p_end, target in anchors:
    s = out[f'{short}_q']
    mins.append(s.min()); maxs.append(s.max()); means.append(s.mean())
    targets.append(target)
    short_labels.append(label.split(' (')[0])

for i in range(len(anchors)):
    ax2.plot([positions[i], positions[i]], [mins[i], maxs[i]],
             color=colors[i % len(colors)], lw=10, alpha=0.45,
             solid_capstyle='butt')
    ax2.plot(positions[i], means[i], 'o',
             color=colors[i % len(colors)], markersize=10,
             markeredgecolor='white', markeredgewidth=1.5,
             label='full-period mean' if i == 0 else None)
    ax2.plot(positions[i], targets[i], '*',
             color='black', markersize=14,
             label='anchor target' if i == 0 else None)
    ax2.annotate(f'{targets[i]:.1f}%',
                 (positions[i], targets[i]),
                 textcoords='offset points', xytext=(10, -3),
                 fontsize=8)

ax2.set_xticks(positions)
ax2.set_xticklabels(short_labels, rotation=20, ha='right', fontsize=8)
ax2.set_ylabel('Shadow economy (% of GDP)', fontsize=10)
ax2.set_title('Implied range under each anchor (bar = quarterly min–max, ● = mean, ★ = anchor target)',
              fontsize=10, pad=10)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax2.grid(alpha=0.3, axis='y')
ax2.set_ylim(15, max(maxs) + 5)
ax2.legend(loc='lower right', fontsize=8)
ax2.set_xlim(-0.7, len(anchors) - 0.3)

plt.tight_layout()
plt.savefig('../outputs/interm_mimic/shadow_economy_anchors_chart.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved shadow_economy_anchors_chart.png")

Saved shadow_economy_anchors.csv  (47 rows, 14 cols)

=== Summary by anchor (quarterly series) ===
  KIIS 2017–2018 (38.4%)
    period 2017Q1-2018Q4: target 38.40%, achieved 38.40%
    full-period mean  : 40.3%
    full-period range : [33.7%, 52.4%]
  NBU 2018 (23.8%)
    period 2018Q1-2018Q4: target 23.80%, achieved 23.80%
    full-period mean  : 24.6%
    full-period range : [18.1%, 36.7%]
  Hassan & Schneider 2010 (59.49%)
    period 2010Q2-2010Q4: target 59.49%, achieved 59.49%
    full-period mean  : 58.9%
    full-period range : [52.4%, 71.0%]
  Hassan & Schneider 2011 (53.60%)
    period 2011Q1-2011Q4: target 53.60%, achieved 53.60%
    full-period mean  : 57.7%
    full-period range : [51.1%, 69.8%]
  Hassan & Schneider 2012 (54.24%)
    period 2012Q1-2012Q4: target 54.24%, achieved 54.24%
    full-period mean  : 60.3%
    full-period range : [53.7%, 72.4%]
  Hassan & Schneider 2013 (53.53%)
    period 2013Q1-2013Q4: target 53.53%, achieved 53.53%
    full-period mean  : 56.3%


# Hybrid CDA+MIMIC

In [45]:
@dataclass
class MIMICResult:
    causes: list
    indicators: list
    n_obs: int
    gamma: np.ndarray
    lambda_: np.ndarray
    psi: float
    theta: np.ndarray
    phi: np.ndarray
    gamma_se: np.ndarray
    lambda_se: np.ndarray
    chi2: float
    df: int
    pvalue: float
    cfi: float
    tli: float
    rmsea: float
    aic: float
    bic: float
    loglik: float
    converged: bool


def _build_sigma(gamma, lam, psi, theta, phi):
    p, q = len(lam), len(gamma)
    var_eta = float(gamma @ phi @ gamma + psi)
    Sigma_yy = np.outer(lam, lam) * var_eta + np.diag(theta)
    Sigma_yx = np.outer(lam, phi @ gamma)
    Sigma = np.zeros((p + q, p + q))
    Sigma[:p, :p] = Sigma_yy
    Sigma[:p, p:] = Sigma_yx
    Sigma[p:, :p] = Sigma_yx.T
    Sigma[p:, p:] = phi
    return Sigma


def _ml_fit_function(Sigma, S):
    sign_S, logdet_S = np.linalg.slogdet(S)
    sign_Si, logdet_Si = np.linalg.slogdet(Sigma)
    if sign_Si <= 0 or sign_S <= 0:
        return 1e10
    try:
        Sigma_inv = np.linalg.inv(Sigma)
    except np.linalg.LinAlgError:
        return 1e10
    k = S.shape[0]
    return logdet_Si + np.trace(S @ Sigma_inv) - logdet_S - k


def fit_mimic(data, causes, indicators, n_starts=8, random_state=0):
    """ML fit, lambda_1 fixed at 1, all variables standardised to z."""
    rng = np.random.default_rng(random_state)
    cols = list(indicators) + list(causes)
    df = data[cols].dropna().copy()
    n = len(df)
    df = (df - df.mean()) / df.std(ddof=1)
    p, q = len(indicators), len(causes)
    Y, X = df[indicators].values, df[causes].values
    Z = np.column_stack([Y, X]) - np.column_stack([Y, X]).mean(axis=0)
    S = (Z.T @ Z) / n
    phi = S[p:, p:]

    def neg(par):
        gamma = par[:q]
        lam_free = par[q:q + (p - 1)]
        log_psi = par[q + (p - 1)]
        log_theta = par[q + p:q + 2 * p]
        lam = np.concatenate([[1.0], lam_free])
        return _ml_fit_function(
            _build_sigma(gamma, lam, np.exp(log_psi), np.exp(log_theta), phi),
            S)

    best = None
    for k in range(n_starts):
        if k == 0:
            g0 = np.linalg.lstsq(X, Y.mean(axis=1), rcond=None)[0]
            lf0 = np.ones(p - 1)
            lp0 = np.log(0.5)
            lt0 = np.log(np.maximum(np.diag(S)[:p] * 0.5, 1e-3))
        else:
            g0 = rng.normal(0, 0.3, q)
            lf0 = rng.normal(1, 0.3, p - 1)
            lp0 = np.log(rng.uniform(0.2, 0.8))
            lt0 = np.log(rng.uniform(0.2, 0.8, p))
        x0 = np.concatenate([g0, lf0, [lp0], lt0])
        try:
            r = optimize.minimize(neg, x0, method='BFGS',
                                  options={'maxiter': 1000, 'gtol': 1e-6})
        except Exception:
            continue
        if best is None or r.fun < best.fun:
            best = r

    gamma_hat = best.x[:q]
    lam_free = best.x[q:q + (p - 1)]
    lam_hat = np.concatenate([[1.0], lam_free])
    psi_hat = float(np.exp(best.x[q + (p - 1)]))
    theta_hat = np.exp(best.x[q + p:q + 2 * p])

    cov = (2.0 / n) * np.asarray(best.hess_inv)
    ses = np.sqrt(np.maximum(np.diag(cov), 1e-12))
    g_se = ses[:q]
    lam_se = np.concatenate([[0.0], ses[q:q + (p - 1)]])

    F_ML = best.fun
    chi2 = n * F_ML
    n_distinct = (p + q) * (p + q + 1) / 2
    n_free = q + (p - 1) + 1 + p + q * (q + 1) / 2
    df_model = max(int(round(n_distinct - n_free)), 1)
    pvalue = 1 - stats.chi2.cdf(chi2, df_model) if chi2 > 0 else 1.0

    diag_S = np.diag(np.diag(S))
    F_base = (np.log(np.linalg.det(diag_S))
              + np.trace(S @ np.linalg.inv(diag_S))
              - np.log(np.linalg.det(S)) - (p + q))
    chi2_base = n * F_base
    df_base = (p + q) * (p + q - 1) / 2
    a, b = max(chi2 - df_model, 0), max(chi2_base - df_base, 0)
    cfi = 1 - a / b if b > 0 else np.nan
    tli = ((chi2_base / df_base - chi2 / df_model) /
           (chi2_base / df_base - 1)) if df_base > 0 else np.nan
    rmsea = np.sqrt(max(chi2 - df_model, 0) / (df_model * n)) if df_model > 0 else np.nan

    Sigma_hat = _build_sigma(gamma_hat, lam_hat, psi_hat, theta_hat, phi)
    loglik = -0.5 * n * ((p + q) * np.log(2 * np.pi)
                          + np.linalg.slogdet(Sigma_hat)[1]
                          + np.trace(S @ np.linalg.inv(Sigma_hat)))
    npar_aic = q + (p - 1) + 1 + p
    aic = -2 * loglik + 2 * npar_aic
    bic = -2 * loglik + npar_aic * np.log(n)

    return MIMICResult(list(causes), list(indicators), n,
                       gamma_hat, lam_hat, psi_hat, theta_hat, phi,
                       g_se, lam_se,
                       chi2, df_model, pvalue, cfi, tli, rmsea,
                       aic, bic, loglik, best.success)


def fit_cda_mimic_hybrid(panel: pd.DataFrame,
                          cda_series: pd.Series,
                          causes, indicators,
                          align_index: bool = True,
                          orient_signs: dict | None = None):
    """
    Step 1-2: take a CDA shadow-economy series as given (in % of GDP).
    Step 3-4: fit the MIMIC; reverse-standardise its latent path so that
              its sample mean and variance equal those of the CDA series.

    Parameters
    ----------
    panel        : DataFrame containing all causes and indicators.
    cda_series   : Series indexed by year_quarter, in % of GDP.
    causes       : list of cause column names in `panel`.
    indicators   : list of indicator column names; first is the
                   reference (loading fixed at 1).
    align_index  : if True, restrict everything to the intersection of
                   panel rows that have all variables AND a CDA value.
    orient_signs : dict {variable: expected_sign} used to choose +1 vs
                   -1 for the latent.  If None, eta is oriented so its
                   correlation with the CDA series is positive.

    Returns
    -------
    dict with the MIMIC fit, the standardised latent, the hybrid path,
    and a summary table.
    """
    if align_index:
        common = panel.index.intersection(cda_series.index)
        panel = panel.loc[common]
        cda_series = cda_series.loc[common]


    res = fit_mimic(panel, causes, indicators, n_starts=10, random_state=0)

    Xz = panel[causes].copy()
    Xz = (Xz - Xz.mean()) / Xz.std(ddof=1)
    eta_z = pd.Series(Xz.values @ res.gamma, index=panel.index, name='eta_z')

    if orient_signs is not None:
        flip_candidates = []
        for fl in (+1, -1):
            ok = sum(np.sign(res.lambda_[i] * fl) ==
                      np.sign(orient_signs.get(indicators[i], 1))
                      for i in range(len(indicators)))
            flip_candidates.append((ok, fl))
        flip = max(flip_candidates)[1]
    else:
        rho = np.corrcoef(eta_z, cda_series)[0, 1]
        flip = +1 if rho >= 0 else -1
    eta_z_oriented = eta_z * flip

    cda_mean = float(cda_series.mean())
    cda_std  = float(cda_series.std(ddof=1))
    e_mean = float(eta_z_oriented.mean())
    e_std  = float(eta_z_oriented.std(ddof=1))
    se_hybrid = (eta_z_oriented - e_mean) / e_std * cda_std + cda_mean
    se_hybrid.name = 'SE_hybrid'

    table = pd.DataFrame({
        'CDA': cda_series,
        'eta_z': eta_z_oriented,
        'SE_hybrid': se_hybrid,
    })

    diagnostics = {
        'cda_mean': cda_mean,
        'cda_std': cda_std,
        'cda_min': float(cda_series.min()),
        'cda_max': float(cda_series.max()),
        'hybrid_mean': float(se_hybrid.mean()),
        'hybrid_std':  float(se_hybrid.std(ddof=1)),
        'hybrid_min':  float(se_hybrid.min()),
        'hybrid_max':  float(se_hybrid.max()),
        'corr_eta_cda': float(np.corrcoef(eta_z_oriented, cda_series)[0, 1]),
        'corr_hybrid_cda': float(np.corrcoef(se_hybrid, cda_series)[0, 1]),
        'flip': flip,
        'mimic_chi2': res.chi2, 'mimic_df': res.df, 'mimic_p': res.pvalue,
        'mimic_cfi': res.cfi, 'mimic_rmsea': res.rmsea,
    }

    return {'mimic': res, 'eta_z': eta_z_oriented,
            'cda': cda_series, 'hybrid': se_hybrid,
            'table': table, 'diagnostics': diagnostics}


if __name__ == '__main__':

    df = pd.read_csv('../data/final_data.csv')
    df = df.set_index('year_quarter')
    cda_df = pd.read_csv('../outputs/shadow_economy_results.csv')
    cda_df = cda_df.set_index('year_quarter')

    CAUSES     = ['d_tax_burden', 'd_rule_of_law', 'd_direct_tax_share']
    INDICATORS = ['d_employment_rate', 'dlog_gdp_s21']

    SIGNS = {
        'd_employment_rate': -1,
        'dlog_gdp_s21':      -1,
    }

    out = fit_cda_mimic_hybrid(
        panel        = df,
        cda_series   = cda_df['shadow_method2'],
        causes       = CAUSES,
        indicators   = INDICATORS,
        orient_signs = SIGNS,
    )

    print("=" * 70)
    print("HYBRID CDA + MIMIC ESTIMATE FOR UKRAINE 2010Q1-2021Q4")
    print("(Dybka, Kowalczuk, Olesiński, Rozkrut & Toroj 2019 procedure)")
    print("=" * 70)

    print("\nMIMIC fit (winning spec from the previous classical run):")
    res = out['mimic']
    print(f"  chi2({res.df}) = {res.chi2:.2f}  p = {res.pvalue:.3f}")
    print(f"  CFI = {res.cfi:.3f}  RMSEA = {res.rmsea:.3f}  AIC = {res.aic:.1f}")
    print(f"  N = {res.n_obs}")

    diag = out['diagnostics']
    print("\nCDA series (column 'shadow_method2'):")
    print(f"  mean = {diag['cda_mean']:.3f}   std = {diag['cda_std']:.3f}")
    print(f"  range = [{diag['cda_min']:.3f}, {diag['cda_max']:.3f}]")

    print("\nHybrid series (after reverse standardisation):")
    print(f"  mean = {diag['hybrid_mean']:.3f}   std = {diag['hybrid_std']:.3f}")
    print(f"  range = [{diag['hybrid_min']:.3f}, {diag['hybrid_max']:.3f}]")

    print("\nDiagnostics:")
    print(f"  corr(eta, CDA)        = {diag['corr_eta_cda']:+.3f}")
    print(f"  corr(hybrid, CDA)     = {diag['corr_hybrid_cda']:+.3f}  "
          "(must equal corr(eta,CDA) since hybrid is affine in eta)")
    print(f"  orientation flip      = {diag['flip']:+d}")

    print("\n--- TIME SERIES (every 4th quarter shown) ---")
    print(out['table'].iloc[::4].round(3).to_string())

    out['table'].to_csv('../outputs/hybrid_estimates.csv')
    print("\nSaved hybrid_estimates.csv")

HYBRID CDA + MIMIC ESTIMATE FOR UKRAINE 2010Q1-2021Q4
(Dybka, Kowalczuk, Olesiński, Rozkrut & Toroj 2019 procedure)

MIMIC fit (winning spec from the previous classical run):
  chi2(2) = 1.49  p = 0.474
  CFI = 1.000  RMSEA = 0.000  AIC = 653.0
  N = 47

CDA series (column 'shadow_method2'):
  mean = 4.003   std = 2.127
  range = [0.000, 8.922]

Hybrid series (after reverse standardisation):
  mean = 4.003   std = 2.127
  range = [-1.069, 11.516]

Diagnostics:
  corr(eta, CDA)        = +0.002
  corr(hybrid, CDA)     = +0.002  (must equal corr(eta,CDA) since hybrid is affine in eta)
  orientation flip      = -1

--- TIME SERIES (every 4th quarter shown) ---
                CDA  eta_z  SE_hybrid
year_quarter                         
2010Q2        6.820 -0.221      2.690
2011Q2        6.276 -0.053      3.686
2012Q2        6.871 -0.105      3.378
2013Q2        4.008 -0.325      2.071
2014Q2        2.707 -0.312      2.150
2015Q2        3.626 -0.209      2.762
2016Q2        4.939 -0.290     

# Hybrid plot

In [46]:
t = pd.read_csv('../outputs/hybrid_estimates.csv',
                index_col=0)
 
 
def quarter_to_date(q):
    y, qn = q.split('Q')
    month = {'1': 2, '2': 5, '3': 8, '4': 11}[qn]
    return pd.Timestamp(year=int(y), month=month, day=15)
 
 
dates = pd.DatetimeIndex([quarter_to_date(q) for q in t.index])
 
fig, axes = plt.subplots(2, 1, figsize=(11.5, 9),
                         gridspec_kw={'height_ratios': [3, 2]})
 
ax = axes[0]
ax.plot(dates, t['CDA'].values, color='#c0504d', lw=2.0,
        label='CDA (input, shadow_method2)', alpha=0.9, marker='o', markersize=3)
ax.plot(dates, t['SE_hybrid'].values, color='#1f4e79', lw=2.4,
        label='Hybrid CDA+MIMIC (Dybka et al. 2019)')
 
hyb_ma = t['SE_hybrid'].rolling(4, min_periods=1).mean()
cda_ma = t['CDA'].rolling(4, min_periods=1).mean()
ax.plot(dates, cda_ma.values, color='#c0504d', lw=1.0, ls='--', alpha=0.6,
        label='CDA, 4-quarter MA')
ax.plot(dates, hyb_ma.values, color='#1f4e79', lw=1.0, ls='--', alpha=0.6,
        label='Hybrid, 4-quarter MA')
 
ax.axhline(t['CDA'].mean(), color='gray', ls=':', lw=0.8)
ax.text(dates[1], t['CDA'].mean() + 0.15,
        f'common mean = {t["CDA"].mean():.2f}%', fontsize=8, color='gray')
 
ax.set_ylabel('Shadow economy (% of GDP)', fontsize=10)
ax.set_title('Hybrid CDA+MIMIC vs CDA-only path\n'
             '(reverse standardisation enforces identical mean and standard deviation)',
             fontsize=11, pad=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=1))
ax.grid(alpha=0.3, axis='y')
ax.legend(loc='upper right', fontsize=8, frameon=True)
ax.set_xlim(dates.min(), dates.max())
ax.axhline(0, color='black', lw=0.5)
 
neg = t['SE_hybrid'] < 0
if neg.any():
    for d, v in zip(dates[neg], t['SE_hybrid'][neg]):
        ax.scatter(d, v, color='red', s=40, zorder=5,
                   marker='x', label='_nolegend_')
    ax.text(dates[neg][0], -0.7,
            f'{neg.sum()} quarter(s) with implausible negative value',
            color='red', fontsize=8)
 
ax2 = axes[1]
ax2.scatter(t['CDA'], t['SE_hybrid'], color='#1f4e79', alpha=0.6, s=30)
lo = min(t['CDA'].min(), t['SE_hybrid'].min())
hi = max(t['CDA'].max(), t['SE_hybrid'].max())
ax2.plot([lo, hi], [lo, hi], color='gray', ls='--', lw=1, label='45° line')
 
slope, intercept = np.polyfit(t['CDA'], t['SE_hybrid'], 1)
xs = np.array([lo, hi])
ax2.plot(xs, slope * xs + intercept, color='#c0504d', lw=1.5,
         label=f'OLS fit: y = {slope:.2f}x + {intercept:.2f}')
 
corr = t['CDA'].corr(t['SE_hybrid'])
ax2.text(0.03, 0.92,
         f'corr(CDA, hybrid) = {corr:+.3f}\n'
         f'CDA mean = hybrid mean = {t["CDA"].mean():.2f}%\n'
         f'CDA std  = hybrid std  = {t["CDA"].std(ddof=1):.2f}',
         transform=ax2.transAxes, fontsize=9, va='top',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                   edgecolor='lightgray'))
 
ax2.set_xlabel('CDA estimate (% of GDP)', fontsize=10)
ax2.set_ylabel('Hybrid CDA+MIMIC estimate (% of GDP)', fontsize=10)
ax2.set_title('Cross-plot: hybrid vs CDA',
              fontsize=11, pad=10)
ax2.grid(alpha=0.3)
ax2.legend(loc='lower right', fontsize=8)
 
plt.tight_layout()
out_path = '../outputs/hybrid_chart.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved {out_path}')

Saved ../outputs/hybrid_chart.png
